In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
# from pyspark import pipelines as dp


In [0]:
dbutils.widgets.text("catalog", "workspace", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

GPS_POSITIONS_SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"
ROUTES = f"{CATALOG}.{SCHEMA}.silver_routes_scd"
VEHICLES = f"{CATALOG}.{SCHEMA}.silver_vehicles_scd"

DIM_VEHICLE = f"{CATALOG}.{SCHEMA}.dim_vehicle"
DIM_ROUTE = f"{CATALOG}.{SCHEMA}.dim_route"
DIM_DESTINATION = f"{CATALOG}.{SCHEMA}.dim_destination"
DIM_TIME = f"{CATALOG}.{SCHEMA}.dim_time"

In [0]:
df_gps = spark.read.table(GPS_POSITIONS_SILVER)
df_routes = (
    spark.read.table(ROUTES)
    .filter(F.col("is_current") == True)
)

df_vehicles = (
    spark.read.table(VEHICLES)
    .filter(F.col("is_current") == True)
)

In [0]:
dim_vehicle = (
    df_vehicles.select("vehicleCode","transportationType", "vehicleCharacteristics","brand", "model", "productionYear", "length", "seats", "standingPlaces", "floorHeight", "driveType", "carrier", "airConditioning", "wheelchairsRamp","ticketMachine", "usb" )
    .dropDuplicates(["vehicleCode"])
    .withColumn("vehicle_key", F.xxhash64("vehicleCode") )
)

In [0]:
(
    dim_vehicle
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DIM_VEHICLE)
)

In [0]:
dim_route= (
    df_routes.select(F.col("route_id").cast("int").alias("route_id"), "route_short_name", "route_type", "route_color", "route_text_color")
    .dropDuplicates(["route_id"])
    .withColumn("route_key", F.xxhash64("route_id"))
)

In [0]:
(
    dim_route
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DIM_ROUTE)
)

In [0]:
dim_destination = (
    df_gps.select("headsign")
    .filter(F.col("headsign").isNotNull())
    .dropDuplicates(["headsign"])
    .withColumn("destination_key", F.xxhash64("headsign"))
)

In [0]:
(
    dim_destination
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DIM_DESTINATION)
)

In [0]:
dim_time = (
    df_gps.filter(F.col("event_time_local").isNotNull())
    .select(F.to_date("event_time_local").alias("date"),
            F.year("event_time_local").alias("year"),
            F.month("event_time_local").alias("month"),
            F.dayofmonth("event_time_local").alias("day"),
            F.dayofweek("event_time_local").alias("day_of_week"),
            F.hour("event_time_local").alias("hour"))
    .dropDuplicates(["date", "hour"])
    .withColumn("is_weekend", F.col("day_of_week").isin(1,7))
    .withColumn("time_key", F.xxhash64("date", "hour"))
)

In [0]:
(
    dim_time
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DIM_TIME)
)